In [16]:
import itertools
import matplotlib.pyplot as plt
import random
from collections import defaultdict

counter = itertools.count(start=0, step=1)

In [ ]:
class Environment():
    """
    Puzzle grid environment. Contains a grid with block objects that can be moved until the red block is at its goal state.
    """
    def __init__(self, size, blocks):
        """Initializes grid with specified size and places the red block."""
        self.puzzle = [[0 for j in range(size[1])] for i in range(size[0])]
        self.goal = (2, 5)
        self.blocks = {"red": self.add_block(blocks[0], red=True)}
        global counter
        counter = itertools.count(start=0, step=1)
        for block_p in blocks[1:]:
            block = self.add_block(block_p)
            self.blocks[block.name] = block

    def add_block(self, positions, red=False):
        if red:
            block = RedBlock(positions)
        else:
            block = Block(positions)
        for i, j in positions:
            self.puzzle[i][j] = block
        return block

    def move(self, block_name, direction):
        block = self.blocks[block_name]
        old_pos = block.positions
        new_pos = [self._move(direction, i, j) for i, j in old_pos]
        n_rows, n_cols = len(self.puzzle), len(self.puzzle[0])
        for new_i, new_j in new_pos:
            if not (0 <= new_i < n_rows and 0 <= new_j < n_cols):
                return False
            if self.puzzle[new_i][new_j] not in [0, block]:
                return False
        for i, j in old_pos:
            self.puzzle[i][j] = 0
        for new_i, new_j in new_pos:
            self.puzzle[new_i][new_j] = block
        block.positions = new_pos
        return self.get_state()

    def get_state(self):
        return tuple((name, tuple(block.positions)) for name, block in self.blocks.items())

    def check_win(self):
        target = self.puzzle[self.goal[0]][self.goal[1]]
        if target != 0:
            if target.name == "red":
                return True
        return False

    def visualize(self):
        n_rows, n_cols = len(self.puzzle), len(self.puzzle[0])

        fig, ax = plt.subplots()
        ax.add_patch(plt.Rectangle((0, 0), n_cols, n_rows, facecolor="white", edgecolor="none"))

        for block in self.blocks.values():
            rows = [i for i, j in block.positions]
            cols = [j for i, j in block.positions]
            i0, i1 = min(rows), max(rows)
            j0, j1 = min(cols), max(cols)
            facecolor = "red" if block.name == "red" else "gray"
            ax.add_patch(plt.Rectangle(
                (j0, i0), j1 - j0 + 1, i1 - i0 + 1,
                facecolor=facecolor,
                edgecolor="black",
                linewidth=1.5,
            ))

        ax.add_patch(plt.Rectangle(
            (0, 0), n_cols, n_rows,
            facecolor="none",
            edgecolor="black",
            linewidth=1.5,
        ))

        ax.set_xlim(0, n_cols)
        ax.set_ylim(n_rows, 0)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.show()

    def _move(self, direction, i, j):
        if direction == "l":
            return i, j - 1
        elif direction == "r":
            return i, j + 1
        elif direction == "u":
            return i - 1, j
        elif direction == "d":
            return i + 1, j


class Block():
    def __init__(self, positions):
        self.name = next(counter)
        self.positions = positions
        if positions[0][0] == positions[1][0]:
            self.orientation = "h"
        elif positions[0][1] == positions[1][1]:
            self.orientation = "v"


class RedBlock():
    def __init__(self, positions):
        self.name = "red"
        self.positions = positions
        self.orientation = "h"


In [18]:
class Agent():
    def __init__(self, blocks, alpha=0.1, gamma=0.95, epsilon=1):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        action_space = [(block, direction) for direction in "lrud" for block in blocks]
        self.qtable = defaultdict(lambda: {action: 0 for action in action_space})

    def policy(self, state):
        actions = self.qtable[state]
        if self.epsilon > random.randint(0, 1):
            return max(actions, key=actions.get)
        else:
            return random.choice(list(actions.keys()))

    def q_learning(self, s0, s1, a, r):
        q1 = self.qtable[s1]
        rpe = r + self.gamma * max(q1.values()) - self.qtable[s0][a]
        self.qtable[s0][a] += self.alpha * rpe


In [ ]:
class Train():
    def __init__(self, agent, initial):
        self.agent = agent
        self.initial = initial

    def solve(self, max_steps=10000, decay=0.99):
        environment = Environment((6, 6), self.initial)
        solved = False
        while not solved and max_steps > 0:
            s0 = environment.get_state()
            a = self.agent.policy(s0)
            s1 = environment.move(a[0], a[1])
            solved = environment.check_win()
            self.agent.q_learning(s0, s1, a, solved)
            self.agent.epsilon *= decay
            max_steps -= 1
        
    def learn(self, n):
        for _ in range(n):
            self.solve()

    def results(self):
        print(self.agent.qtable)

In [20]:
puzzle_1 = [[(2, 0), (2, 1)], [(0, 1), (1, 1)], [(2, 2), (3, 2)], [(1, 4), (2, 4)], [(4, 0), (5, 0)], [(4, 3), (4, 4), (4, 5)]]
puzzle = Environment((6, 6), puzzle_1)
agent = Agent(puzzle.blocks)


In [21]:
trainer = Train(agent, puzzle_1)
trainer.learn(1000)

(0, 'd')
(4, 'd')
(1, 'r')
('red', 'l')
('red', 'd')
(4, 'l')
('red', 'l')
('red', 'l')
('red', 'l')
(0, 'r')
('red', 'l')
('red', 'l')
('red', 'l')
('red', 'l')
('red', 'l')
('red', 'l')
(2, 'd')
('red', 'd')
('red', 'l')
('red', 'l')
('red', 'r')
('red', 'd')
(0, 'u')
(3, 'r')
(0, 'u')
('red', 'l')
('red', 'l')
('red', 'l')
(0, 'l')
(4, 'd')
('red', 'l')
('red', 'l')
(1, 'u')
('red', 'l')
(1, 'u')
('red', 'l')
(1, 'd')
('red', 'l')
(4, 'd')
(0, 'l')
(4, 'u')
(3, 'u')
(4, 'd')
('red', 'l')


IndexError: list index out of range